In [1]:
import os
import urllib.request
if not os.path.exists("the-verdict.txt"):#如果文件不存在则创建，防止因文件已存在而报错
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

### 实现字节对编码——BPE

In [2]:
import importlib
import tiktoken

In [3]:
tokenizer = tiktoken.get_encoding("gpt2")

在实际开发中更常见的方式是按模式自动匹配tokenizer
例如：enc = tiktoken.encoding_for_model("gpt-4o")

### 利用滑动窗口进行数据采集
对于每个文本块，我们需要输入和目标。由于我们希望模型预测下一个单词，因此我们生成的目标是将输入右移一个位置后的单词。

In [4]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)#读入了一个text并编码到enc_text里面
print(len(enc_text))

5145


In [5]:
enc_sample = enc_text[50:]#从第五十一个开始向后
context_size = 4 #sliding windows4

x = enc_sample[:context_size]#开头四个
y = enc_sample[1:context_size+1]#第二个开始的四个

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [6]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.10.0


/Users/dyj/Downloads/llm/.venv/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


创建数据集Dataset和数据加载器DataLoader，从输入文本数据集中提取文本块

把原始文本转换成可以用于训练 GPT 的训练样本（input–target token 序列），实现一个PyTorch Dataset，并使用 **滑动窗口（sliding window）**从长文本中生成训练数据。

In [7]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    #让GPT初始化一个类型
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize 文本，把文本转换为token id
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # 使用滑动窗口生成训练数据
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [8]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

使用批次大小为1，上下文大小为4的设置，测试dataloader的表现

In [9]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)#数据加载器 dataloader 转换为一个迭代器
first_batch = next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


### 创建标记嵌入（token embeddings）+位置编码

In [10]:
vocab_size = 50257#字节对编码器的词汇表达小
output_dim = 256#假设将输入标记编码为256维的向量
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)#映射为tensor


In [11]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
token_embeddings = token_embedding_layer(inputs)#调用token_embedding_layer将输入inputs映射为对应的嵌入向量。
print(token_embeddings.shape)

torch.Size([8, 4, 256])


gpt2使用绝对位置嵌入，因此只需要创建另一个位置嵌入层

In [12]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
#目的是为输入序列中的每个位置生成一个向量,表明位置信息
pos_embeddings = pos_embedding_layer(torch.arange(max_length))#生成一个连续整数的1D tensor
print(pos_embeddings.shape)

torch.Size([4, 256])


In [13]:
input_embeddings = token_embeddings + pos_embeddings#特征是词语信息跟位置信息的结合
print(input_embeddings.shape)

torch.Size([8, 4, 256])


## 注意力机制

In [14]:
import torch.nn as nn

In [15]:
class CausalSelfAttention(nn.Module):
    """
    该类实现了因果自注意力机制（Causal Self Attention），
    用于自回归模型（例如GPT模型中的注意力层）。
    """

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        """
        初始化因果自注意力层。
        
        参数：
        - d_in: 输入维度
        - d_out: 输出维度
        - context_length: 上下文长度（即注意力机制能“看到”的最大令牌数）
        - dropout: Dropout率
        - qkv_bias: 是否为查询、键和值使用偏置（默认为False）
        """
        super().__init__()
        self.d_out = d_out
        # 定义查询、键、值的线性变换
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
        # Dropout层
        self.dropout = nn.Dropout(dropout)  # 新增的Dropout层

        # 注册一个buffer，用于存储因果掩码
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))  # 新增掩码，禁止未来的信息

    def forward(self, x):
        """
        前向传播函数计算因果自注意力输出。
        
        参数：
        - x: 输入的张量，形状为 (batch_size, num_tokens, d_in)
        
        返回：
        - context_vec: 自注意力机制的输出，形状为 (batch_size, num_tokens, d_out)
        """
        b, n_tokens, d_in = x.shape  # 获取输入张量的维度
        keys = self.W_key(x)  # 键（K）
        queries = self.W_query(x)  # 查询（Q）
        values = self.W_value(x)  # 值（V）

        # 计算注意力分数（查询和键的点积）
        attn_scores = queries @ keys.transpose(1, 2)  # 这里的转置（transpose）是为了匹配维度
        
        # 使用掩码阻止未来的tokens看到当前token
        attn_scores.masked_fill_(  # 这里的操作是原地修改
            self.mask.bool()[:n_tokens, :n_tokens], -torch.inf)  # 将掩码区域填充为负无穷

        # 计算注意力权重并进行softmax归一化
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)  # 使用Dropout层

        # 计算上下文向量（加权和）
        context_vec = attn_weights @ values
        return context_vec


class MultiHeadAttentionWrapper(nn.Module):
    """
    该类实现了多头注意力机制（Multi-Head Attention）包装器。
    它包含多个因果自注意力头，并在输出时将多个头的结果合并。
    """

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        """
        初始化多头注意力层。
        
        参数：
        - d_in: 输入维度
        - d_out: 输出维度
        - context_length: 上下文长度
        - dropout: Dropout率
        - num_heads: 注意力头的数量
        - qkv_bias: 是否为查询、键和值使用偏置（默认为False）
        """
        super().__init__()
        # 定义多个因果自注意力头
        self.heads = nn.ModuleList(
            [CausalSelfAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]  # 为每个头创建一个CausalSelfAttention实例
        )
        
        # 定义最终的线性变换，用于将多个头的输出合并
        self.out_proj = nn.Linear(d_out * num_heads, d_out * num_heads)

    def forward(self, x):
        """
        前向传播函数，计算多头注意力输出。
        
        参数：
        - x: 输入的张量，形状为 (batch_size, num_tokens, d_in)
        
        返回：
        - out: 多头注意力的输出，形状为 (batch_size, num_tokens, d_out * num_heads)
        """
        # 将多个头的输出拼接在一起
        context_vec = torch.cat([head(x) for head in self.heads], dim=-1)
        
        # 通过线性变换得到最终输出
        return self.out_proj(context_vec)

In [16]:
torch.manual_seed(123)

context_length = max_length
d_in = output_dim

num_heads=2
d_out = d_in // num_heads

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads)

batch = input_embeddings
context_vecs = mha(batch)

print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([8, 4, 256])


### 另一种变体

In [17]:
class MultiHeadAttention(nn.Module):
    """
    该类实现了多头自注意力机制（Multi-Head Attention），
    用于自回归模型（如Transformer和GPT中的注意力层）。
    """

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        """
        初始化多头自注意力层。
        
        参数：
        - d_in: 输入维度
        - d_out: 输出维度
        - context_length: 上下文长度（即注意力机制能“看到”的最大令牌数）
        - dropout: Dropout率
        - num_heads: 注意力头的数量
        - qkv_bias: 是否为查询、键和值使用偏置（默认为False）
        """
        super().__init__()

        # 检查输出维度是否能被头数整除
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # 将输出维度除以头数，得到每个头的维度

        # 定义查询、键、值的线性变换
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # 定义输出的线性变换层，用于合并多个头的输出
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        # 注册一个buffer，用于存储因果掩码
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))  # 新增掩码，禁止未来的信息

    def forward(self, x):
        """
        前向传播函数，计算多头自注意力输出。
        
        参数：
        - x: 输入张量，形状为 (batch_size, num_tokens, d_in)
        
        返回：
        - context_vec: 多头自注意力的输出，形状为 (batch_size, num_tokens, d_out)
        """
        b, num_tokens, d_in = x.shape  # 获取输入张量的维度

        # 计算键、查询和值的表示
        keys = self.W_key(x)  # 键（K）
        queries = self.W_query(x)  # 查询（Q）
        values = self.W_value(x)  # 值（V）

        # 将最后的维度按头数进行拆分：
        # 将 (b, num_tokens, d_out) 转换为 (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # 转置以适配矩阵相乘：
        # (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # 计算缩放点积注意力（self-attention）
        attn_scores = queries @ keys.transpose(2, 3)  # 点积计算每个头的注意力分数

        # 通过掩码将未来的信息遮掩（变成负无穷）
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]  # 将掩码转换为布尔类型
        attn_scores.masked_fill_(mask_bool, -torch.inf)  # 使用掩码将未来的信息填充为负无穷

        # 计算注意力权重并进行softmax归一化
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)  # 使用Dropout层

        # 计算上下文向量（加权和）
        context_vec = (attn_weights @ values).transpose(1, 2)  # 恢复维度 (b, num_tokens, num_heads, head_dim)

        # 合并头部的输出，并进行线性变换
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)  # 合并头的输出
        context_vec = self.out_proj(context_vec)  # 可选的投影层

        return context_vec

In [18]:
torch.manual_seed(123)

context_length = max_length
d_in = output_dim
d_out = d_in
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

batch = input_embeddings
context_vecs = mha(batch)

print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([8, 4, 256])


### 归一化层
将神经网络层的激活值中心化为均值为 0，并将其方差归一化为 1。
这种方法能够稳定训练过程，并加速权重的高效收敛。
在 Transformer 块中，层归一化会在多头注意力模块的前后应用（我们将在后续实现），并在最终输出层之前再次应用。

In [19]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5  #避免0的产生导致崩溃
        self.scale = nn.Parameter(torch.ones(emb_dim))#动态的缩放参数
        self.shift = nn.Parameter(torch.zeros(emb_dim)) #动态的偏移参数

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)#算平均值
        var = x.var(dim=-1, keepdim=True, unbiased=False)#算方差
        norm_x = (x - mean) / torch.sqrt(var + self.eps)#归一化
        return self.scale * norm_x + self.shift#通过Ω和 œ 调整归一化后的值范围和位置

*缩放与平移*
除了通过减去均值并除以方差来执行归一化操作外，我们还引入了两个可训练参数：scale（缩放参数）和 shift（平移参数）。
初始时，scale 值为 1，shift 值为 0，不会对结果产生影响；但在训练过程中，LLM 会自动调整这两个参数，以提升模型在任务中的表现。
这种设计使模型能够学习到最适合其数据的缩放和平移方式。
此外，在计算方差的平方根时，我们会添加一个较小的值（eps），以避免方差为 0 时出现除以 0 的错误。

GELU作为激活函数

In [20]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

FeedForward（前馈网络）：对每个token的表示进行非线性特征变化，从而增加模型表达能力
Attention 负责 token 之间的信息交互
FeedForward 负责对每个 token 做深度特征变换
在这里遵循gpt2的版本使用linear-gelu- linear

In [21]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

### 在 Transformer 块中连接attention层与线性层
transformer块中还包含dropout和残差连接机制（输出 = 原始信息 + 学到的修正）
残差连接是为了防止深层网络退化（梯度消失，训练困难等），改善梯度传播，保留原始信息

In [22]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],# 输入特征维度
            d_out=cfg["emb_dim"],# 输出特征维度
            context_length=cfg["context_length"],# 上下文长度
            num_heads=cfg["n_heads"],# 注意力头的数量
            dropout=cfg["drop_rate"],# Dropout 比例
            qkv_bias=cfg["qkv_bias"])# 查询、键和值的偏置
        self.ff = FeedForward(cfg)# 前馈神经网络模块
        self.norm1 = LayerNorm(cfg["emb_dim"])# 第一归一化层
        self.norm2 = LayerNorm(cfg["emb_dim"])# 第二归一化层
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"]) # 残差连接的 Dropout

    def forward(self, x):
        # 对注意力模块的快捷连接（残差连接）
        shortcut = x
        x = self.norm1(x)#应用第一归一化层
        x = self.att(x)   # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)#应用dropout
        x = x + shortcut  # 将原始输入加回去，实现残差连接

        # Shortcut connection for feed-forward block对前馈网络模块的残差连接
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x


logits = 模型对每个词的“未归一化概率分数”。

In [23]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        #新建字典、位置信息、还有dropout的比率设置
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])#归一化
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):

        # 每次生成一个单词后，重新将其加入序列中
        # 如果当前上下文长度超过模型支持的最大上下文长度，则截取
        # 例如，如果LLM只支持5个token，而上下文长度为10
        # 那么只使用最后5个token作为上下文
        idx_cond = idx[:, -context_size:]#如果idx的长度超过模型支持的上下文长度size，只保留最后size个token

        # Get the predictions
        with torch.no_grad(): # 在推理阶段，不需要计算梯度，因为没有反向传播。减少储存开销
            logits = model(idx_cond)#模型输出结果

        logits = logits[:, -1, :]#只关注最后一个时间步的输出

        # 获取最高概率值的词汇索引
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx


def main():
    GPT_CONFIG_124M = {
        "vocab_size": 50257,     # Vocabulary size
        "context_length": 1024,  # Context length
        "emb_dim": 768,          # Embedding dimension
        "n_heads": 12,           # Number of attention heads
        "n_layers": 12,          # Number of layers
        "drop_rate": 0.1,        # Dropout rate
        "qkv_bias": False        # Query-Key-Value bias
    }

    torch.manual_seed(123)
    model = GPTModel(GPT_CONFIG_124M)
    model.eval()  # disable dropout

    start_context = "Hello, I am"

    tokenizer = tiktoken.get_encoding("gpt2")
    encoded = tokenizer.encode(start_context)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)

    print(f"\n{50*'='}\n{22*' '}IN\n{50*'='}")
    print("\nInput text:", start_context)
    print("Encoded input text:", encoded)
    print("encoded_tensor.shape:", encoded_tensor.shape)

    out = generate_text_simple(
        model=model,
        idx=encoded_tensor,
        max_new_tokens=10,
        context_size=GPT_CONFIG_124M["context_length"]
    )
    decoded_text = tokenizer.decode(out.squeeze(0).tolist())

    print(f"\n\n{50*'='}\n{22*' '}OUT\n{50*'='}")
    print("\nOutput:", out)
    print("Output length:", len(out[0]))
    print("Output text:", decoded_text)


if __name__ == "__main__":
    main()


                      IN

Input text: Hello, I am
Encoded input text: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


                      OUT

Output: tensor([[15496,    11,   314,   716, 27018, 24086, 47843, 30961, 42348,  7267,
         49706, 43231, 47062, 34657]])
Output length: 14
Output text: Hello, I am Featureiman Byeswickattribute argue logger Normandy Compton analogous


为什么只关注最后一个时间步？——语言模型只需要预测“下一个 token”。

In [33]:
%pip install numpy tensorflow tqdm tiktoken torch

  Using cached numpy-2.4.2-cp311-cp311-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.34.0-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached keras-3.13.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached wheel-0.46.3-py3-none-any.whl.metadata (2.4 kB)
  Using cached rich-14.3.3-py3-none-any.whl.metadata (18 kB)
  Using cached namex-

## 评估文本生成大模型

In [43]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();

In [42]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor
#给输入的字符进行编码并实现一个Batch维度的向量,符合模型的输入形式
def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())
#反向编码,去掉移除张量中的批次维度, 变成普通的链表

In [44]:
start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    #初始上下文的Token ID张量，是上一步 text_to_token_ids 的输出
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)
#输出最长单词度为10的句子
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


### 计算文本生成loss ：交叉熵和困惑度（cross-entropy，perplexity）

In [48]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]
#用向量的形式展现输入的文本
targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]
#用向量的形式展现要输出的东西

In [49]:
with torch.no_grad():
    logits = model(inputs)
#不用梯度计算的计算inputes并储存
probas = torch.softmax(logits, dim=-1) # Probability of each token in vocabulary
#用soft Max整理logits
print(probas.shape) # Shape: (batch_size, num_tokens, vocab_size)每个token位置都有一个对整个词表的预测

torch.Size([2, 3, 50257])


In [50]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
#相当于用贪心算法给出最有可能的答案
print("Token IDs:\n", token_ids)

Token IDs:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])


In [51]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
#给出答案
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")
#给出事实上的结论

Targets batch 1:  effort moves you
Outputs batch 1:  Armed heNetflix


以上结果是由于模型还没有经过训练，要训练模型我们需要知道他与正确预测之间的差距有多大

In [52]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:", target_probas_2)
#从模型预测的概率probas中取出模型给正确答案的概率

Text 1: tensor([7.4541e-05, 3.1061e-05, 1.1563e-05])
Text 2: tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])


probas[text_idx, [0,1,2], targets[text_idx]]就是对某个句子在每个token位置，取出模型给正确token的概率。因此我们希望最大化这些值，使他们的概率接近1

In [53]:
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
avg_log_probas = torch.mean(log_probas)
neg_avg_log_probas = avg_log_probas * -1
#最大化对数等价为最小化负对数
print(neg_avg_log_probas)

tensor(10.7940)


在深度学习中，通常的做法是最小化 "负" 的平均对数概率值，而不是最大化平均对数概率值；在我们的例子中，深度学习中我们会最小化 10.7722 使其接近 0，而不是最大化 -10.7722 使其接近 0。
值 -10.7722 的负数，即 10.7722，在深度学习中也被称为交叉熵损失（cross-entropy loss）

In [54]:
#更少代码的实现计算loss的方法
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
#为了配合cross entropy需要的输入格式
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

tensor(10.7940)


In [55]:
perplexity = torch.exp(loss)
#指数化loss作为P值
print(perplexity)

tensor(48725.8203)


困惑度通常被认为更易解释，因为它可以理解为模型在每一步对词汇表大小的不确定性（在上面的例子中，这相当于 48,725 个单词或 token）。
换句话说，困惑度提供了一个衡量模型预测的概率分布与数据集中单词实际分布匹配程度的指标。
类似于损失值，较低的困惑度表示模型预测与实际分布的差距较小。

## 计算训练集和验证集的损失

In [56]:
import os
import urllib.request

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
#引入数据集
if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

In [58]:
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]
torch.manual_seed(123)#保持结果可复现
train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)
#初始化输入训练模型,给出批处理的大小、给出最大文本容量防止溢出
#给出不畅,丢弃最后一批不足的文本,打开随机防止拟合过度


val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)
#验证数据集仅仅修改了是否丢弃跟随抽取


In [59]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    #用交叉熵函数对于logits进行计算并且拉伸到二维长度
    return loss
#一个计算批损失的函数

In [60]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # 如果指定的批次数超过数据加载器中的总批次数，则将批次数减少到与数据加载器的总批次数匹配。
        num_batches = min(num_batches, len(data_loader))
        #减少需要处理的数量,同时也是防止溢出
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
        #一点点加上去的损失
    return total_loss / num_batches

In [82]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [62]:
model.to(device)  # 对于 nn.Module 类，不需要赋值 model = model.to(device)
torch.manual_seed(123)
with torch.no_grad():  # 禁用梯度跟踪以提高效率，因为此时尚未开始训练
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)
    # 推理阶段不计算梯度
print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 10.987582948472765
Validation loss: 10.98110580444336


## 训练llm

In [63]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    #初始化训练模型而且给了空的队列
    # Main training loop
    for epoch in range(num_epochs):#训练次数
        model.train()  # Set model to training mode
        #转移到训练模块
        for input_batch, target_batch in train_loader:
            #从loader里面调出输入跟目标
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            #清空所有函数的梯度
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            #计算损失函数
            loss.backward() # Calculate loss gradients
            #反向传播优化
            optimizer.step() # Update model weights using loss gradients
            #更新权重
            tokens_seen += input_batch.numel()
            #加一下一共有多少
            global_step += 1
            #看一下一共训练了多少步
            # Optional evaluation step
            if global_step % eval_freq == 0:
                #按照一定的步数进行记录
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                #计算损失函数
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                #加到list中
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Print a sample text after each epoch
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    #评价模块
    model.eval()
    #检验模式
    with torch.no_grad():
        #双保险,防止梯度更新
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    #	在评估结束后切换回训练模式，确保模型能继续用于训练。
    return train_loss, val_loss

def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()


global_step 是训练循环中的重要计数器，主要用于控制学习率调度、记录日志、保存模型检查点和控制终止条件等任务。
在分批训练中，它提供了一个统一的参考点，有助于管理复杂的训练流程。
epoch 是按完整数据集的迭代单位，而 global_step 是按 batch 单位，粒度更细，用于管理更精确的任务。例如：
动态学习率调整,某些学习率调度器需要以 batch 为单位进行调整，而不是每个 epoch。例如，WarmUp 会在固定的前 N 步逐渐升高学习率。
频繁日志记录,记录训练日志时，通常是每隔 log_interval 步输出一次，而不是每个 epoch。
检查点保存,保存模型状态通常是按步数完成，尤其是当训练需要中断和恢复时，global_step 是更精确的 token。

In [64]:
import time
start_time = time.time()#看一下用了多久
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
#用Adam进行优化,其中学习rate为0.004,动量衰减是0.1
num_epochs = 10
#10论学习
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"训练完成耗时 {execution_time_minutes:.2f} 分钟。")

Ep 1 (Step 000000): Train loss 9.817, Val loss 9.924
Ep 1 (Step 000005): Train loss 8.066, Val loss 8.332
Every effort moves you,,,,,,,,,,,,.                                     
Ep 2 (Step 000010): Train loss 6.619, Val loss 7.042
Ep 2 (Step 000015): Train loss 6.046, Val loss 6.596
Every effort moves you, and,, and, and,,,,, and, and,,,,,,,,,,, and,, the,, the, and,, and,,, the, and,,,,,,
Ep 3 (Step 000020): Train loss 5.524, Val loss 6.508
Ep 3 (Step 000025): Train loss 5.369, Val loss 6.378
Every effort moves you, and to the of the of the picture. Gis.                                     
Ep 4 (Step 000030): Train loss 4.830, Val loss 6.263
Ep 4 (Step 000035): Train loss 4.586, Val loss 6.285
Every effort moves you of the "I the picture.                    "I"I the picture"I had the picture"I the picture and I had been the picture of
Ep 5 (Step 000040): Train loss 3.879, Val loss 6.130
Every effort moves you know he had been his pictures, and I felt it's by his last word.          

### 控制随机性的解码策略

在之前的实现中，我们始终使用 torch.argmax 来选择概率最高的 token 作为下一个生成的 token。
为了增加生成文本的多样性，我们可以改用 torch.multinomial(probs, num_samples=1)，从概率分布中随机采样下一个 token。
在这种方法中，每个索引被选中的概率与其在输入张量中对应的概率值成正比，从而实现基于概率的随机采样。

In [72]:
vocab = { 
    "closer": 0,
    "every": 1, 
    "effort": 2, 
    "forward": 3,
    "inches": 4,
    "moves": 5, 
    "pizza": 6,
    "toward": 7,
    "you": 8,
} 

inverse_vocab = {v: k for k, v in vocab.items()}
#插入
# Suppose input is "every effort moves you", and the LLM
# returns the following logits for the next token:
next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

probas = torch.softmax(next_token_logits, dim=0)
#softmax归一化
next_token_id = torch.argmax(probas).item()
#选个可能性最大
# The next generated token is then as follows:
print(inverse_vocab[next_token_id])

forward


如果不再依赖 torch.argmax 来选择最可能的 token ，而是通过 torch.multinomial(probas, num_samples=1) 从 softmax 分布中采样来确定下一个 token：

In [73]:
torch.manual_seed(123)
next_token_id = torch.multinomial(probas, num_samples=1).item()
print(inverse_vocab[next_token_id])

forward


In [74]:
#温度缩放
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)
#温度校正
# Temperature values
temperatures = [1, 0.1, 5]  # Original, higher confidence, and lower confidence
#初始校正系数
# Calculate scaled probabilities
scaled_probas = [softmax_with_temperature(next_token_logits, T) for T in temperatures]

*Top-k取样*
为了在使用更高温度增加输出多样性的同时减少生成无意义句子的概率，我们可以将采样限制在前 k 个最可能的 token 中

In [75]:
top_k = 3
top_logits, top_pos = torch.topk(next_token_logits, top_k)
#topK采样
print("Top logits:", top_logits)
print("Top positions:", top_pos)

Top logits: tensor([6.7500, 6.2800, 4.5100])
Top positions: tensor([3, 7, 0])


In [76]:
new_logits = torch.where(
    condition=next_token_logits < top_logits[-1],
    input=torch.tensor(float("-inf")), 
    other=next_token_logits
)
#不是前K遮蔽掉
print(new_logits)

tensor([4.5100,   -inf,   -inf, 6.7500,   -inf,   -inf,   -inf, 6.2800,   -inf])


In [77]:
topk_probas = torch.softmax(new_logits, dim=0)
print(topk_probas)

tensor([0.0615, 0.0000, 0.0000, 0.5775, 0.0000, 0.0000, 0.0000, 0.3610, 0.0000])


- 温度校正是更加平滑,防止数据差之毫厘以谬以千里 
- topK是防止臭鱼烂虾进入筛选范围提高质量

现在，我们将结合这两种方法，对之前用于生成大语言模型（LLM）文本的 generate_simple 函数进行改进，创建一个新的 generate 函数

In [78]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
#生成模块
    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        #计算预测值,但是切最后一个
        # New: Filter logits with top_k sampling
        #top K采样
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)
            
        # New: Apply temperature scaling
        #温度校正
        if temperature > 0.0:
            logits = logits / temperature

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)
            #从概率分布中采样下一个 token 

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)
            #如果未启用采样，选择概率最高的 token 作为下一个 token 
        if idx_next == eos_id:  # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

###  在Pytorch中加载并保留权重

In [83]:
torch.save(model.state_dict(), "model.pth")

In [84]:
model = GPTModel(GPT_CONFIG_124M)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
model.eval();

In [85]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    }, 
    "model_and_optimizer.pth"
)

In [86]:
checkpoint = torch.load("model_and_optimizer.pth", weights_only=True)
#保存检查点
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();
#调整到训练模式

## 从openai导入超参数

In [ ]:
%pip install transformers

In [89]:
from transformers import GPT2Model


# allowed model names
model_names = {
    "gpt2-small (124M)": "openai-community/gpt2",
    "gpt2-medium (355M)": "openai-community/gpt2-medium",
    "gpt2-large (774M)": "openai-community/gpt2-large",
    "gpt2-xl (1558M)": "openai-community/gpt2-xl"
}

CHOOSE_MODEL = "gpt2-small (124M)"

gpt_hf = GPT2Model.from_pretrained(model_names[CHOOSE_MODEL], cache_dir="checkpoints")
gpt_hf.eval()

/Users/dyj/Downloads/llm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 13624.45it/s]


GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [90]:
BASE_CONFIG = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "drop_rate": 0.0,       # Dropout rate
    "qkv_bias": True        # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}


BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [91]:
def assign_check(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(right.clone().detach())

In [92]:
import numpy as np


def load_weights(gpt, gpt_hf):

    d = gpt_hf.state_dict()
    gpt.pos_emb.weight = assign_check(gpt.pos_emb.weight, d["wpe.weight"])#位置权重
    gpt.tok_emb.weight = assign_check(gpt.tok_emb.weight, d["wte.weight"])#单词权重
    
    for b in range(BASE_CONFIG["n_layers"]):
        q_w, k_w, v_w = np.split(d[f"h.{b}.attn.c_attn.weight"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign_check(gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign_check(gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign_check(gpt.trf_blocks[b].att.W_value.weight, v_w.T)
    
        q_b, k_b, v_b = np.split(d[f"h.{b}.attn.c_attn.bias"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign_check(gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign_check(gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign_check(gpt.trf_blocks[b].att.W_value.bias, v_b)
    
    
        gpt.trf_blocks[b].att.out_proj.weight = assign_check(gpt.trf_blocks[b].att.out_proj.weight, d[f"h.{b}.attn.c_proj.weight"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign_check(gpt.trf_blocks[b].att.out_proj.bias, d[f"h.{b}.attn.c_proj.bias"])
    
        gpt.trf_blocks[b].ff.layers[0].weight = assign_check(gpt.trf_blocks[b].ff.layers[0].weight, d[f"h.{b}.mlp.c_fc.weight"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign_check(gpt.trf_blocks[b].ff.layers[0].bias, d[f"h.{b}.mlp.c_fc.bias"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign_check(gpt.trf_blocks[b].ff.layers[2].weight, d[f"h.{b}.mlp.c_proj.weight"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign_check(gpt.trf_blocks[b].ff.layers[2].bias, d[f"h.{b}.mlp.c_proj.bias"])
    
        gpt.trf_blocks[b].norm1.scale = assign_check(gpt.trf_blocks[b].norm1.scale, d[f"h.{b}.ln_1.weight"])
        gpt.trf_blocks[b].norm1.shift = assign_check(gpt.trf_blocks[b].norm1.shift, d[f"h.{b}.ln_1.bias"])
        gpt.trf_blocks[b].norm2.scale = assign_check(gpt.trf_blocks[b].norm2.scale, d[f"h.{b}.ln_2.weight"])
        gpt.trf_blocks[b].norm2.shift = assign_check(gpt.trf_blocks[b].norm2.shift, d[f"h.{b}.ln_2.bias"])
    
        gpt.final_norm.scale = assign_check(gpt.final_norm.scale, d[f"ln_f.weight"])
        gpt.final_norm.shift = assign_check(gpt.final_norm.shift, d[f"ln_f.bias"])
        gpt.out_head.weight = assign_check(gpt.out_head.weight, d["wte.weight"])

In [93]:
import torch
gpt = GPTModel(BASE_CONFIG)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
load_weights(gpt, gpt_hf)

In [94]:
import tiktoken

torch.manual_seed(123)

tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate(
    model=gpt.to(device),
    idx=text_to_token_ids("Every effort moves", tokenizer).to(device),
    max_new_tokens=30,
    context_size=BASE_CONFIG["context_length"],
    top_k=1,
    temperature=1.0
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves forward, but it's not enough.

"I'm not going to sit here and say, 'I'm not going to do this,'


### 使用垃圾邮件和非垃圾邮件组成的数据集对llm进行分类微调

In [95]:
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"
#完成导入数据
def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return

    # 下载文件
    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    # 解压文件
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)
    # 添加.tsv文件扩展名
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

File downloaded and saved as sms_spam_collection/SMSSpamCollection.tsv


In [ ]:
%pip install pandas

In [97]:
import pandas as pd
df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [98]:
print(df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


需要解决数据类别不平衡的问题（class imbalance）
为什么需要解决？
模型可能会学到偷懒策略，永远预测ham（数量多）

In [99]:
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    #算一下类别为spam的样本出现的次数
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    #随机采样 “ham” 使其数量与“spam”样本数量一致
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])
    #合并 “spam” 和采样后的 “ham”数据
    return balanced_df


balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [100]:
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})  

In [101]:
def random_split(df, train_frac, validation_frac):
    # 把数据集Dataframe打乱
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    # 计算分割系数
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    # 分割数据集Dataframe
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
# 剩余部分为测试集，占总数据集的比例为 0.2

train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)
#DataFrame 被随机划分为训练集、验证集和测试集并保存

In [102]:
from torch.utils.data import Dataset

class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)
        #先读入文本
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]
        #编码成对应的词元id
        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            #如果序列长度超过 max_length，则进行截断
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # 填充到最长序列的长度
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]
    def __getitem__(self, index):
        #获取指定索引的数据样本
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )
    def __len__(self):
        return len(self.data)
        #输出序列长度
    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length

统一所有数据集的序列长度，长度由训练集决定，避免data leakage

In [103]:
train_dataset = SpamDataset(
    csv_file="train.csv",
    max_length=None,
    tokenizer=tokenizer
)

print(train_dataset.max_length)

120


In [104]:
val_dataset = SpamDataset(
    csv_file="validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
test_dataset = SpamDataset(
    csv_file="test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
#将验证集和测试集数据填充到最长序列的长度

In [105]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

# 设置种子确保可复现
torch.manual_seed(123)
# 初始化数据加载器
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)
# 对齐超参数,但是dataset需要区别一下训练集、验证集和测试集

In [106]:
print("Train loader:")
for input_batch, target_batch in train_loader:
    pass
    #如果这个数据在训练集出现过,则跳过

print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

Train loader:
Input batch dimensions: torch.Size([8, 120])
Label batch dimensions torch.Size([8])


In [107]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

130 training batches
19 validation batches
38 test batches


### 添加分类头

In [108]:
for param in model.parameters():
    param.requires_grad = False
# 返回 model 中所有的参数,但是不参与反向传播

In [109]:
torch.manual_seed(123)

num_classes = 2
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)

In [110]:
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True
# 对于需要更新权重则设将该参数设置为True
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print("Inputs:", inputs)
print("Inputs dimensions:", inputs.shape)
# 编码并输出

Inputs: tensor([[5211,  345,  423,  640]])
Inputs dimensions: torch.Size([1, 4])


In [111]:
with torch.no_grad():
    outputs = model(inputs)

print("Outputs:\n", outputs)
print("Outputs dimensions:", outputs.shape)

Outputs:
 tensor([[[ 0.2156,  0.2835],
         [-0.0921, -0.4722],
         [ 0.7829, -0.3221],
         [ 1.6524, -0.4964]]])
Outputs dimensions: torch.Size([1, 4, 2])


### 计算分类损失和准确率

In [112]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0
    # 先改成评估模式,在初始化两个储存器
    if num_batches is None:
        num_batches = len(data_loader)
        # num_batches设为None则取全部数据
    else:
        num_batches = min(num_batches, len(data_loader))
        # 取num_batches和len(data_loader)中较小的那个
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)
            # 仅处理前 num_batches 批次的数据
            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]  # 最后一个输出词元的概率
            # 预测内容取概率最大值
            predicted_labels = torch.argmax(logits, dim=-1)
            # 当前批次的样本数
            num_examples += predicted_labels.shape[0]
            # 预测正确的样本数量
            correct_predictions += (predicted_labels == target_batch).sum().item()
            
        else:
            break
    return correct_predictions / num_examples

In [113]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device) # 对于 nn.Module 类，无需 model = model.to(device)赋值操作

torch.manual_seed(123) # 由于训练数据加载器中的随机打乱，因此设置随机种子以确保可重复性

train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)
#各种计算
print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Training accuracy: 53.75%
Validation accuracy: 55.00%
Test accuracy: 51.25%


In [114]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]  # Logits of last output token
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss
    #用交叉熵计算损失函数

In [115]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # 如果 num_batches 超过数据加载器中的批次数，则减少批次数以匹配数据加载器中的总批次数
        # num_batches = min(num_batches, len(data_loader))
        for i, (input_batch, target_batch) in enumerate(data_loader):
            if i < num_batches:
                loss = calc_loss_batch(input_batch, target_batch, model, device)
                # 总损失值
                total_loss += loss.item()
            
            else:
                break
    return total_loss / num_batches

In [116]:
with torch.no_grad(): # 因为我们不进行训练，所以为了提高效率禁用梯度跟踪
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")

Training loss: 0.984
Validation loss: 0.938
Test loss: 1.012


### 在有监督数据上微调模型

In [117]:
def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                            eval_freq, eval_iter):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1
    # 初始化分类头
    for epoch in range(num_epochs):
        model.train()  
        # 每次都进入训练模块
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() 
            # 清零梯度
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() 
            # 对损失值执行反向传播记录梯度
            optimizer.step() 
            # 用梯度优化权重
            examples_seen += input_batch.shape[0] 
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"Training accuracy: {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

In [118]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [119]:
import time

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

num_epochs = 5
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Ep 1 (Step 000000): Train loss 0.998, Val loss 0.845
Ep 1 (Step 000050): Train loss 0.518, Val loss 0.394
Ep 1 (Step 000100): Train loss 0.292, Val loss 0.380
Training accuracy: 85.00% | Validation accuracy: 85.00%
Ep 2 (Step 000150): Train loss 0.441, Val loss 0.335
Ep 2 (Step 000200): Train loss 0.297, Val loss 0.339
Ep 2 (Step 000250): Train loss 0.269, Val loss 0.259
Training accuracy: 92.50% | Validation accuracy: 82.50%
Ep 3 (Step 000300): Train loss 0.062, Val loss 0.297
Ep 3 (Step 000350): Train loss 0.112, Val loss 0.304
Training accuracy: 92.50% | Validation accuracy: 87.50%
Ep 4 (Step 000400): Train loss 0.052, Val loss 0.408
Ep 4 (Step 000450): Train loss 0.048, Val loss 0.315
Ep 4 (Step 000500): Train loss 0.088, Val loss 0.248
Training accuracy: 97.50% | Validation accuracy: 90.00%
Ep 5 (Step 000550): Train loss 0.038, Val loss 0.423
Ep 5 (Step 000600): Train loss 0.014, Val loss 0.260
Training accuracy: 100.00% | Validation accuracy: 92.50%
Training completed in 4.40 min

In [120]:
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Training accuracy: 99.52%
Validation accuracy: 93.96%
Test accuracy: 91.67%


### 用llm作为垃圾消息分类器

In [121]:
def classify_review(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    model.eval()
    # 将模型设置为评估模式，禁用dropout等训练特定操作
    input_ids = tokenizer.encode(text)
    # 使用tokenizer将输入文本 转换token ID 列表
    supported_context_length = model.pos_emb.weight.shape[0]
    # 获取模型支持的最大上下文长度（由位置嵌入的权重数量决定）
    input_ids = input_ids[:min(max_length, supported_context_length)]
    # 如果 token 序列长度超过模型支持的最大长度，则进行截断
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    # 对序列进行填充
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0) 
    # 将 token ID 转换为张量并移动到指定设备（如 GPU），同时增加 batch 维度
    # 模型推理
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :] 
        # 禁用梯度计算，仅进行推理
        # 将输入张量传入模型，获取 logits，并提取最后一个 token 的 logits
    predicted_label = torch.argmax(logits, dim=-1).item()
    # 在 logits 上取 softmax 概率最大值的索引，作为预测类别
    return "spam" if predicted_label == 1 else "not spam"
    # 如果预测标签是 `1`，返回 "spam"，否则返回 "not spam"

In [122]:
#试一下
text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)

print(classify_review(
    text_1, model, tokenizer, device, max_length=train_dataset.max_length
))

spam


In [123]:
#保存模型
torch.save(model.state_dict(), "review_classifier.pth")

## 指令微调

In [124]:
#为有监督指令微调准备数据集
def download_and_load_file(file_path, url):

    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data
# 在网上下载并打开数据库

file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))
# 看一下数据一共有多少条

Number of entries: 1100


我们默认使用Alpaca风格的提示格式，这是一种较早公开并被广泛使用的指令微调提示模板

In [125]:
def format_input(entry):
    # 使用数据库的提示词
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    
    # 如果没有输入的格式，将如何处理
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    
    return instruction_text + input_text

In [126]:
# 自定义训练集、测试集和验证集的大小
train_portion = int(len(data) * 0.85)  # 85% 作为训练集
test_portion = int(len(data) * 0.1)    # 10% 作为测试集
val_portion = len(data) - train_portion - test_portion  # 剩下的 5% 作为验证集

# 划分数据集
train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

In [127]:
print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


In [128]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    # 指示数据类的构建
    def __init__(self, data, tokenizer):
        self.data = data
        # 实例化数据
        # 对文本进行预编码
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )
    # 链表访问
    def __getitem__(self, index):
        return self.encoded_texts[index]
    
    def __len__(self):
        return len(self.data)

在之前，我们将数据集中的所有例子填充为相同的长度。
而在这里，我们采取了一种更为复杂的方法: 开发了一个自定义的 "collate" 函数，并将其传递给数据加载器。
这个自定义的collate函数会将每个批次中的训练示例填充到相同的长度（不同批次的长度可以不同）。

In [129]:
def custom_collate_draft_1(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # 找到批次中最长的序列
    # 并将最大长度增加1，这样会在后面添加一个额外的填充 token
    batch_max_length = max(len(item)+1 for item in batch)

    inputs_lst = []
    # 准备输入
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        # 复制后进行填充
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        # 去掉最后一个表示并保存
        inputs = torch.tensor(padded[:-1])
        
        inputs_lst.append(inputs)
    # 堆积起来并输送给gpu
    inputs_tensor = torch.stack(inputs_lst).to(device)
    
    return inputs_tensor

In [130]:
def custom_collate_draft_2(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # 找到最大的序列长度
    batch_max_length = max(len(item)+1 for item in batch)
    # 准备一个空列表
    inputs_lst, targets_lst = [], []
    
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        # 输入值是第一个到倒数第二个
        inputs = torch.tensor(padded[:-1]) 
        # 目标值是第二个到最后一个，这样子保证了长度一样
        targets = torch.tensor(padded[1:]) 
        
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [131]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    # 找到批次中最长的序列
    batch_max_length = max(len(item)+1 for item in batch)

    # 填充并准备输入和目标
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # 添加一个 <|endoftext|> token 
        new_item += [pad_token_id]
        # 将序列填充到最大长度
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # 截断最后一个 token 作为输入
        targets = torch.tensor(padded[1:])  # 向右移1个位置作为目标

        # 新增：将目标中除了第一个填充 token 外的所有填充 token 替换为 ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # 新增：根据需要，限制序列的最大长度
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # 将输入和目标的列表转换为张量，并转移到目标设备
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

### 创建指令数据集的dataloader

In [132]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

Device: mps


In [133]:
from functools import partial
# 初始化定义
customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

In [134]:
num_workers = 0
batch_size = 8

torch.manual_seed(123)
# 初始化训练
train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

In [135]:
# 初始化验证与测试
val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [142]:
from gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel, load_weights_into_gpt


BASE_CONFIG = {
    "vocab_size": 50257,     # 词表大小
    "context_length": 1024,  # 上下文长度
    "drop_rate": 0.0,        # Dropout率
    "qkv_bias": True         # 查询-键-值偏置
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(
    model_size=model_size,
    models_dir="gpt2"
)

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 3.29kiB/s]
encoder.json: 100%|██████████| 1.04M/1.04M [00:00<00:00, 1.11MiB/s]
hparams.json: 100%|██████████| 91.0/91.0 [00:00<00:00, 46.8kiB/s]
model.ckpt.data-00000-of-00001: 100%|██████████| 1.42G/1.42G [03:58<00:00, 5.94MiB/s] 
model.ckpt.index: 100%|██████████| 10.4k/10.4k [00:00<00:00, 3.59MiB/s]
model.ckpt.meta: 100%|██████████| 927k/927k [00:00<00:00, 1.00MiB/s]
vocab.bpe: 100%|██████████| 456k/456k [00:00<00:00, 702kiB/s] 


GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_resid): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_f

In [143]:
torch.manual_seed(123)
input_text = format_input(val_data[0])
print(input_text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'


In [144]:
from previous_chapters import (
    generate,
    text_to_token_ids,
    token_ids_to_text
)

token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)

In [145]:
response_text = (
    # 从生成的文本开始计数
    generated_text[len(input_text):]
    #如果生成的文本包含 `### Response:`，则删除它
    .replace("### Response:", "")
    #去掉空格
    .strip()
)
print(response_text)

The chef cooks the meal every day.

### Instruction:

Convert the active sentence to passive: 'The chef cooks the


In [146]:
from previous_chapters import (
    calc_loss_loader,
    train_model_simple
)

In [147]:
model.to(device)

torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

# 先看一次没有微调的结果
print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 3.8259099960327148
Validation loss: 3.7619343757629395


In [148]:
import time

start_time = time.time()

torch.manual_seed(123)

# 用Adam训练,并定义了学习率、权重衰减等参数
optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)

num_epochs = 2

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten(), ignore_index=-100)
    return loss


start_time = time.time()

torch.manual_seed(123)

# 用Adam训练,并定义了学习率、权重衰减等参数
optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)

num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")


Ep 1 (Step 000000): Train loss 2.637, Val loss 2.626
Ep 1 (Step 000005): Train loss 1.174, Val loss 1.103
Ep 1 (Step 000010): Train loss 0.872, Val loss 0.944
Ep 1 (Step 000015): Train loss 0.857, Val loss 0.906
Ep 1 (Step 000020): Train loss 0.776, Val loss 0.881
Ep 1 (Step 000025): Train loss 0.754, Val loss 0.859
Ep 1 (Step 000030): Train loss 0.800, Val loss 0.836
Ep 1 (Step 000035): Train loss 0.714, Val loss 0.809
Ep 1 (Step 000040): Train loss 0.672, Val loss 0.806
Ep 1 (Step 000045): Train loss 0.633, Val loss 0.789
Ep 1 (Step 000050): Train loss 0.663, Val loss 0.783
Ep 1 (Step 000055): Train loss 0.760, Val loss 0.763
Ep 1 (Step 000060): Train loss 0.719, Val loss 0.743


RuntimeError: MPS backend out of memory (MPS allocated: 14.56 GiB, other allocations: 5.39 GiB, max allowed: 20.13 GiB). Tried to allocate 196.32 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [149]:
torch.manual_seed(123)


for entry in test_data[:3]:

    input_text = format_input(entry)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
)

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response:\n>> {response_text.strip()}")
    print("-------------------------------------")

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Rewrite the sentence using a simile.

### Input:
The car is very fast.

Correct response:
>> The car is as fast as lightning.

Model response:
>> The car is very fast.
-------------------------------------
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What type of cloud is typically associated with thunderstorms?

Correct response:
>> The type of cloud typically associated with thunderstorms is cumulonimbus.

Model response:
>> A cloud is typically associated with thunderstorms.
-------------------------------------
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Name the author of 'Pride and Prejudice'.

Correct response:
>> Jane Austen.

Model response:
>> The author of 'Pride and Prejudice' is Sir Thomas More.

In [151]:
from tqdm import tqdm
import json

for i, entry in tqdm(enumerate(test_data), total=len(test_data)):

    input_text = format_input(entry)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = generated_text[len(input_text):].replace("### Response:", "").strip()

    test_data[i]["model_response"] = response_text


with open("instruction-data-with-response.json", "w") as file:
    json.dump(test_data, file, indent=4) 

100%|██████████| 110/110 [03:50<00:00,  2.09s/it]


In [152]:
import re


file_name = f"{re.sub(r'[ ()]', '', CHOOSE_MODEL) }-sft.pth"
torch.save(model.state_dict(), file_name)
print(f"Model saved as {file_name}")

Model saved as gpt2-medium355M-sft.pth


In [154]:
import psutil

def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")
# 检查运行情况
if not ollama_running:
    raise RuntimeError("Ollama not running. Launch ollama before proceeding.")
print("Ollama running:", check_if_running("ollama"))

Ollama running: True


In [156]:
import urllib.request
import json

def query_model(
    prompt,
    model="llama3.1:8b",
    url="http://localhost:11434/api/chat"
):
    # 预处理的类型与符号
    # 创建数据负载字典
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {     # 以下设置是为了获得确定性的响应
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }
    

    # 将字典转换为JSON格式的字符串并编码为字节
    payload = json.dumps(data).encode("utf-8")
    # 对JSON进行编码
    # 创建请求对象，设置请求方法为POST并添加必要的头部信息
    request = urllib.request.Request(
        url,
        data=payload,
        method="POST"
    )
    # 设置请求头部为JSON格式
    request.add_header("Content-Type", "application/json")

    # 发送请求并捕获响应
    response_data = ""
    with urllib.request.urlopen(request) as response:
        # 读取并解码响应
        while True:
            line = response.readline().decode("utf-8")
            if not line:
                break
            response_json = json.loads(line)
            response_data += response_json["message"]["content"]

    return response_data


model = "llama3.1:8b"
result = query_model("What do Llamas eat?", model)
print(result)

Llamas are herbivores, which means they primarily eat plants and plant-based foods. Their diet consists of:

1. **Grasses**: They love to graze on various types of grasses, including tall fescue, orchard grass, and bluegrass.
2. **Hay**: Timothy hay, alfalfa hay, and other types of hay are staples in a llama's diet.
3. **Fruits**: Apples, carrots, and sweet potatoes are all treats that llamas enjoy.
4. **Grains**: Oats, corn, and barley can be given to llamas as supplements or treats.
5. **Leafy greens**: Llamas will eat leafy greens like kale, spinach, and collard greens.

In the wild, llamas would typically roam in herds and graze on a variety of plants, including shrubs and trees. In captivity, their diet is often supplemented with commercial llama feed or pellets to ensure they receive all the necessary nutrients.

Some interesting facts about llama eating habits:

* Llamas have a three-part stomach, similar to cows, which allows them to digest plant material more efficiently.
* Th

In [157]:
for entry in test_data[:3]:
    # 自定义提示词
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry['model_response']}`"
        f" on a scale from 0 to 100, where 100 is the best score. "
    )

    print("\nDataset response:")
    print(">>", entry['output'])
    print("\nModel response:")
    print(">>", entry["model_response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("\n-------------------------")


Dataset response:
>> The car is as fast as lightning.

Model response:
>> The car is very fast.

Score:
>> I'd give the model response "The car is very fast." a score of 20 out of 100.

Here's why:

* The instruction asks for a simile, which is a figure of speech that compares two things using "like" or "as."
* The correct output "The car is as fast as lightning" uses a simile to compare the car's speed to lightning.
* However, the model response simply repeats the original sentence without making any changes. It doesn't use a simile or any other literary device to enhance the language.

Therefore, I'd give it a score of 20 out of 100, indicating that it only partially meets the requirements of the instruction.

-------------------------

Dataset response:
>> The type of cloud typically associated with thunderstorms is cumulonimbus.

Model response:
>> A cloud is typically associated with thunderstorms.

Score:
>> I would score this model response as 20 out of 100.

Here's why:

* The

In [159]:
# 生成模型得分的函数
def generate_model_scores(json_data, json_key, model="llama3.1:8b"):
    scores = []
    
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        score = query_model(prompt, model)
        try:
            scores.append(int(score))
        except ValueError:
            print(f"Could not convert score: {score}")
            continue

    return scores


scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores)/len(scores):.2f}\n")

Scoring entries: 100%|██████████| 110/110 [01:15<00:00,  1.45it/s]

Number of scores: 110 of 110
Average score: 63.63

